In [46]:
code_snippets = [
    {
        "name": "add_auth_token",
        "code": """
def add_auth_token(request, token):
    request.headers["Authorization"] = f"Bearer {token}"
    return request
"""
    },
    {
        "name": "sort_users",
        "code": """
def sort_users(users):
    return sorted(users, key=lambda user: user.name)
"""
    },
    {
        "name": "save_viewport",
        "code": """
def save_viewport(zoom, center_x, center_y):
    url = f"?zoom={zoom}&centerX={center_x}&centerY={center_y}"
    history.push(url)
"""
    },
    {
        "name": "filter_deleted_users",
        "code": """
def filter_deleted_users(users):
    return [user for user in users if not user.deleted]
"""
    }
]

In [47]:
import re


def normalize_text(text):
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    text = text.replace("_", " ")

    return text.lower()

In [48]:
documents = [
    normalize_text(code_snippet["code"]) for code_snippet in code_snippets
]

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(documents)

print(document_vectors.shape)

print(vectorizer.get_feature_names_out())
print(document_vectors.toarray())

(4, 29)
['add' 'auth' 'authorization' 'bearer' 'center' 'def' 'deleted' 'filter'
 'for' 'headers' 'history' 'if' 'in' 'key' 'lambda' 'name' 'not' 'push'
 'request' 'return' 'save' 'sort' 'sorted' 'token' 'url' 'user' 'users'
 'viewport' 'zoom']
[[0.20549991 0.20549991 0.20549991 0.20549991 0.         0.10723838
  0.         0.         0.         0.20549991 0.         0.
  0.         0.         0.         0.         0.         0.
  0.61649973 0.13116793 0.         0.         0.         0.61649973
  0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.14067682
  0.         0.         0.         0.         0.         0.
  0.         0.26957769 0.26957769 0.26957769 0.         0.
  0.         0.17206794 0.         0.26957769 0.26957769 0.
  0.         0.42507629 0.63761443 0.         0.        ]
 [0.         0.         0.         0.         0.8220542  0.071497
  0.         0.         0.         0.         0.13700903 0.
  0.     

In [50]:
query = "where is authentication token added to request"

query_vector = vectorizer.transform([query])

print(query_vector.shape)
print(query_vector.toarray())

(1, 29)
[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.70710678 0.         0.         0.         0.         0.70710678
  0.         0.         0.         0.         0.        ]]


In [51]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(
    query_vector,
    document_vectors
)

print(similarities)
print(similarities.shape)

[[0.87186228 0.         0.         0.        ]]
(1, 4)


In [52]:
scores = similarities[0]
idx = scores.argmax()

print(idx)

best_snippet = code_snippets[idx]

print("Best match:", best_snippet["name"])
print("Score:", scores[idx])
print(best_snippet["code"])

0
Best match: add_auth_token
Score: 0.8718622815548744

def add_auth_token(request, token):
    request.headers["Authorization"] = f"Bearer {token}"
    return request



In [53]:
sorted_indices = scores.argsort(descending=True)[:3]

for i, idx in enumerate(sorted_indices):
    snippet = code_snippets[idx]
    print(f"{i + 1}. {snippet['name']} - {scores[idx]:.4f}")

1. add_auth_token - 0.8719
2. sort_users - 0.0000
3. save_viewport - 0.0000


In [54]:
def search(query, top_k=3):
    # 1. Преобразовать query через уже обученный vectorizer
    query_vector = vectorizer.transform([normalize_text(query)])

    # 2. Рассчитать similarity со всеми document_vectors
    similarities = cosine_similarity(query_vector, document_vectors)

    # 3. Получить одномерный scores
    scores = similarities[0]
    
    # 4. Найти top_k индексов
    sorted_indices = scores.argsort()[::-1][:top_k]

    # 5. Вывести результаты
    for i, idx in enumerate(sorted_indices):
        snippet = code_snippets[idx]
        print(f"{i + 1}. {snippet['name']} - {scores[idx]:.4f}")


In [55]:
document_vectors.shape

(4, 29)

In [56]:
search("sort users alphabetically", top_k=2)
print()

search("store zoom in URL", top_k=2)
print()

search("exclude removed accounts", top_k=2)

1. sort_users - 0.6065
2. filter_deleted_users - 0.3206

1. save_viewport - 0.3955
2. filter_deleted_users - 0.1264

1. filter_deleted_users - 0.0000
2. save_viewport - 0.0000


In [65]:
from exctractor import find_source_files, load_source_files

paths = find_source_files("sample_repository")
source_files = load_source_files(paths)

documents = [
    normalize_text(source_file["code"]) for source_file in source_files 
]

vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(documents)